# Perturbation Analysis

**Updated to use unified analysis framework**

This notebook now uses the unified condition analysis system (`run_condition_analysis.sh`) which handles both disease and perturbation analyses through a single configuration-driven approach.

**Key changes:**
- Use `run_condition_analysis.sh --dataset <name>` instead of old perturbation-specific scripts
- Results are in `base_folder/output/stats/` instead of `base_folder/output/perturbations/`
- Same configuration system for all datasets (disease and perturbation)
- The old `src/feature_association/perturbation/` module is deprecated

**Available perturbation datasets:**
- `op`: OP compounds (control: Dimethyl Sulfoxide)
- `CXCL9`: Ruxolitinib (multiple controls)
- `parsebioscience`: Cytokines (control: PBS)


In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import pandas as pd
plt.rcParams["figure.figsize"]=4,4
warnings.filterwarnings("ignore")
from ciim.src.common import base_dir

In [9]:
!cd ../ && bash scripts/experiment/run_overlap_analysis.sh \
    --dataset parsebioscience \
    --cell-types "CD4T CD8T" \
    --feature-type tf_activity \
    --skip-condition-stats

INTERVENTION OVERLAP ANALYSIS PIPELINE

Configuration:
  Dataset: parsebioscience
  Cell types: CD4T CD8T
  Feature type: tf_activity
  Skip condition stats: true
  Mode: Weighted (centrality + effect sizes)


STEP 1: Skipping condition analysis (using cached stats)

STEP 2: Running overlap analysis...

DRUG REVERSAL ANALYSIS

Dataset: parsebioscience
Feature type: tf_activity
Cell types: ['CD4T', 'CD8T']
Significance threshold: 0.05
Use all drug stats: True
Use weighting (centrality + effect sizes): True

Loading aging statistics...
  Loaded 451 aging TF associations
  Cell types: ['CD4T', 'CD8T', 'NK', 'B', 'MONO']
  Unique TFs: 302

Loading drug statistics...
  Using unified stats: stats_parsebioscience_bulk_tf_activity_mixed-effect.csv
  Loaded 35100 drug TF associations
  Unique drugs/comparisons: 90
  Drugs: ['4-1BBL', 'ADSF', 'APRIL', 'BAFF', 'C3a', 'C5a', 'CD27L', 'CD30L', 'CD40L', 'CT-1', 'Decorin', 'EGF', 'EPO', 'FGF-beta', 'FLT3L', 'FasL', 'G-CSF', 'GDNF', 'GITRL', 'GM-CSF',

In [50]:
# !ls -lt ../base_folder/output/perturbations/

In [12]:
dataset = 'parsebioscience'  # 'CXCL9', 'op', 'parsebioscience'
df_all = pd.read_csv(f'{base_dir}/output/perturbations/reversal_stats_{dataset}.csv')

# Filter significant results
df_sig = df_all[df_all['fisher_pvalue_adj'] < 0.05].copy()

print(f"Dataset: {dataset}")
print(f"Total records: {len(df_all)}")
print(f"Significant features: {len(df_sig)}")
print(f"\nExample - IL-10 in CD4T:")
df_all[(df_all['cell_type']=='CD4T') & (df_all['condition']=='IL-10')].head()

Dataset: parsebioscience
Total records: 180
Significant features: 40

Example - IL-10 in CD4T:


,condition,cell_type,dataset,n_common,total_centrality_weight,avg_tf_centrality,n_reversal,reversal_weight,acceleration_weight,reversal_score,fisher_pvalue,effect_size,a_weighted,b_weighted,c_weighted,d_weighted,fisher_pvalue_adj,classification,effect_type
16,IL-10,CD4T,parsebioscience,51,1.720225,0.213046,1,1.210983,0.509242,0.407936,0.667071,0.185047,0.509242,0.094583,1.1164,0.0,0.926148,neutral,reversal


In [ ]:
!ls -lt 

In [ ]:
# Analysis of perturbation effects
# Note: The old 'reversal_stats' files are from the drug reversal analysis (separate module)
# This shows the basic condition statistics from the unified analysis

dataset = 'parsebioscience'
stats = pd.read_csv(f'{base_dir}/output/stats/stats_{dataset}_bulk_tf_activity_mixed-effect.csv')

# Get significant changes per treatment and cell type
sig_stats = stats[stats['p_value_adj'] < 0.05].copy()
sig_stats['direction'] = sig_stats['slope_condition'].apply(lambda x: 'increase' if x > 0 else 'decrease')

# Summary by treatment
summary = sig_stats.groupby(['condition', 'cell_type', 'direction']).size().reset_index(name='count')
print("Significant TF changes by treatment:")
print(summary.pivot_table(index='condition', columns=['cell_type', 'direction'], values='count', fill_value=0))

# For specific treatment (e.g., IL-10 in CD4T)
print(f"\nIL-10 rejuvenating TFs in CD4T:")
il10_cd4t = sig_stats[(sig_stats['cell_type']=='CD4T') & (sig_stats['condition']=='IL-10')]
print(f"Number of significant TFs: {len(il10_cd4t)}")
il10_cd4t.head()


Rejuvinative drugs in CD4T:  number:  3  list:  ['FLT3L', 'IL-31', 'C5a']


,drug,cell_type,dataset,n_common,total_centrality_weight,avg_tf_centrality,n_reversal,reversal_weight,acceleration_weight,reversal_score,fisher_pvalue,effect_size,a_weighted,b_weighted,c_weighted,d_weighted,fisher_pvalue_adj,classification,effect_type
0,FLT3L,CD4T,parsebioscience,4,0.519722,0.238311,0,0.519722,0.0,1.0,0.0,1.0,0.0,0.158912,0.360810,0.0,0.0,rejuvenating,reversal
1,IL-31,CD4T,parsebioscience,5,0.472488,0.239672,0,0.472488,0.0,1.0,0.0,1.0,0.0,0.000000,0.472488,0.0,0.0,rejuvenating,reversal
2,C5a,CD4T,parsebioscience,5,0.449383,0.190790,0,0.449383,0.0,1.0,0.0,1.0,0.0,0.221512,0.227871,0.0,0.0,rejuvenating,reversal


In [77]:
aging_clocks_rej_drugs = ['Tamatinib',
 '5-(9-Isopropyl-8-methyl-2-morpholino-9H-purin-6-yl)pyrimidin-2-amine',
 'Foretinib',
 'TL_HRAS26',
 'Ruxolitinib',
 'AVL-292',
 'LY2090314',
 'Dasatinib',
 'PF-04691502',
 'Perhexiline',
 'Flutamide',
 'BMS-536924',
 'IKK Inhibitor VII',
 'Idelalisib',
 'PD-0325901',
 'CHIR-99021',
 'Defactinib',
 'BI-D1870',
 'Crizotinib',
 'Nilotinib',
 'Saracatinib',
 'PRT-062607',
 'MGCD-265',
 'R428',
 'Selumetinib',
 'GLPG0634']
df = pd.read_csv(f'{base_dir}/output/perturbations/rejuvenating_drugs_op.csv')
rej_drugs = df[df['cell_type']=='CD4T']['drug'].tolist()
len(set(rej_drugs).intersection(set(aging_clocks_rej_drugs)))

19

- find the common compounds between aging clocks, fisher's test
- repeat for each dataset of op and clcx9
- repeat for promotor based 
- evaluate for cd8t